# Document Type Aware Processing - Interactive Demo

Interactive notebook for testing document-type-specific processing with Phase 3 vision-language models.
Features automatic document detection, schema optimization, and performance comparison.

In [ ]:
#!/usr/bin/env python3
"""
Document Type Aware Processing Notebook - Interactive Phase 3 Demo

Interactive notebook for testing document-type-specific processing with advanced features:
- Automatic document type detection (invoice, bank_statement, receipt)
- Schema-driven field reduction (19-15 fields vs 25 unified)
- Performance comparison and efficiency metrics
- V100-optimized memory management
"""

import sys
from pathlib import Path
from PIL import Image
from IPython.display import display, Markdown, HTML
import time

# Add project root to path (parent directory since we're in notebooks/)
project_root = Path('..').absolute()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"📂 Project root: {project_root}")
print("✅ Document-aware processing environment configured")
print("🎯 Phase 3 capabilities: Document detection + Schema optimization + Performance tracking")

In [ ]:
# Import the document-aware processors (Phase 3)
try:
    from models.llama_processor_v2 import DocumentAwareLlamaProcessor
    from models.internvl3_processor_v2 import DocumentAwareInternVL3Processor
    from common.extraction_parser import discover_images
    print("✅ Document-aware processors imported successfully")
    print("🎯 Using Phase 3 document-type-specific schema system")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("💡 Make sure Phase 3 processors exist: models/*_processor_v2.py")

## Configuration

Edit the cells below to configure your experiment:

In [ ]:
# ============================================================================
# EDIT THIS SECTION FOR YOUR EXPERIMENT
# ============================================================================

# Choose model: "llama" or "internvl3"
MODEL = "llama"

# Document awareness enabled (Phase 3 feature)
ENABLE_DOCUMENT_AWARENESS = True

# Your experimental prompt (edit this!)
# NOTE: If document awareness is enabled, the system will:
# 1. Auto-detect document type (invoice, bank_statement, receipt)
# 2. Use document-specific schema (19-15 fields instead of 25)
# 3. Generate targeted prompts for better efficiency
EXPERIMENTAL_PROMPT = """
Extract key information from this business document in JSON format.
Focus on the most important fields for this document type.
Return clean, structured data with proper field names.
"""

# Test image (or None to use first available)
TEST_IMAGE = "../evaluation_data/synthetic_invoice_001.png"

# Show document detection details? (True/False)
SHOW_DOCUMENT_ANALYSIS = True

# Maximum tokens for generation (increase for longer outputs)
MAX_TOKENS = 2048

print(f"🚀 Configuration:")
print(f"   Model: {MODEL}")
print(f"   Document Awareness: {'✅ Enabled' if ENABLE_DOCUMENT_AWARENESS else '❌ Disabled'}")
print(f"   Max tokens: {MAX_TOKENS}")
print(f"   Image: {TEST_IMAGE or 'auto-select'}")
print(f"   Document analysis: {SHOW_DOCUMENT_ANALYSIS}")

## Display Prompt

In [ ]:
# Display the experimental prompt
display(HTML("<h3>📝 Experimental Prompt:</h3>"))
display(Markdown(f"```\n{EXPERIMENTAL_PROMPT}\n```"))
print(f"\n📊 Prompt statistics:")
print(f"   Length: {len(EXPERIMENTAL_PROMPT)} characters")
print(f"   Lines: {len(EXPERIMENTAL_PROMPT.splitlines())}")
print(f"   Words: {len(EXPERIMENTAL_PROMPT.split())}")

## Load and Display Image

In [ ]:
# Auto-select image if none specified
image_path = TEST_IMAGE
if not image_path:
    try:
        from common.extraction_parser import discover_images
        images = discover_images("../evaluation_data/")
        if images:
            image_path = str(images[0])
            print(f"🎯 Auto-selected: {Path(image_path).name}")
        else:
            print("❌ No test images found in ../evaluation_data/")
    except Exception as e:
        print(f"❌ Error finding images: {e}")

# Load and display the image
if image_path and Path(image_path).exists():
    img = Image.open(image_path)
    
    display(HTML(f"<h3>🖼️ Test Image: {Path(image_path).name}</h3>"))
    
    # Resize for display if too large
    max_display_width = 800
    if img.width > max_display_width:
        ratio = max_display_width / img.width
        new_height = int(img.height * ratio)
        img_display = img.resize((max_display_width, new_height), Image.Resampling.LANCZOS)
        display(img_display)
        print(f"📐 Original size: {img.width}x{img.height}")
        print(f"📐 Display size: {img_display.width}x{img_display.height}")
    else:
        display(img)
        print(f"📐 Image size: {img.width}x{img.height}")
else:
    print(f"❌ Image not found: {image_path}")

## Run Model Processing

In [ ]:
# Initialize the document-aware processor
print(f"🚀 Initializing {MODEL} document-aware processor...")
print(f"🎯 Document awareness: {'Enabled' if ENABLE_DOCUMENT_AWARENESS else 'Disabled'}")

try:
    if MODEL.lower() == "llama":
        processor = DocumentAwareLlamaProcessor(
            debug=True,
            enable_document_awareness=ENABLE_DOCUMENT_AWARENESS
        )
    elif MODEL.lower() == "internvl3":
        processor = DocumentAwareInternVL3Processor(
            debug=True,
            enable_document_awareness=ENABLE_DOCUMENT_AWARENESS
        )
    else:
        raise ValueError(f"Unsupported model: {MODEL}")
    
    print(f"✅ {MODEL} processor initialized successfully")
    if ENABLE_DOCUMENT_AWARENESS:
        print("🎯 Document-type-specific schema system active")
        print("   → Automatic document classification")
        print("   → Targeted field extraction (19-15 fields vs 25)")
        print("   → Optimized prompts for each document type")
    else:
        print("📋 Using unified schema (25 fields)")
        
except Exception as e:
    print(f"❌ Error initializing processor: {e}")
    print("💡 Make sure you're running this on a machine with GPU and model access")
    raise

In [ ]:
# Run document-aware processing
display(HTML("<h3>🔬 Processing with Document-Aware Model...</h3>"))

start_time = time.time()
result = None

try:
    print("🧪 Running document-aware processing...\n")
    
    # Process image with document awareness
    result = processor.process_single_image(image_path)
    
    processing_time = time.time() - start_time
    
    # Extract information from result
    model_response = result.get('raw_response', 'No response available')
    parsed_data = result.get('parsed_data', {})
    
    print(f"\n✅ Processing completed in {processing_time:.2f} seconds")
    print(f"📏 Response length: {len(model_response)} characters")
    
    # Display document awareness results if enabled
    if ENABLE_DOCUMENT_AWARENESS and SHOW_DOCUMENT_ANALYSIS:
        doc_awareness = result.get('document_awareness', {})
        if doc_awareness:
            print("\n🎯 Document Analysis Results:")
            print(f"   Detected type: {result.get('detected_document_type', 'unknown')}")
            print(f"   Fields extracted: {result.get('fields_extracted', 'unknown')}")
            print(f"   Field reduction: {result.get('field_reduction', 0)} fields")
            print(f"   Efficiency gain: {result.get('efficiency_gain', '0%')}")
            print(f"   Processing mode: {doc_awareness.get('processing_mode', 'unknown')}")
    
except Exception as e:
    print(f"❌ Error during processing: {e}")
    import traceback
    traceback.print_exc()
    model_response = None
    parsed_data = {}

## Display Results

In [ ]:
# Display raw model output
if model_response:
    display(HTML("<h3>🤖 Raw Model Output:</h3>"))
    display(HTML(f'<div style="background-color: #f0f0f0; padding: 10px; border-radius: 5px; font-family: monospace; white-space: pre-wrap; max-height: 400px; overflow-y: auto;">{model_response}</div>'))
else:
    print("❌ No model response to display")

In [ ]:
# Display rendered markdown (if applicable)
if model_response and "markdown" in EXPERIMENTAL_PROMPT.lower():
    display(HTML("<h3>📄 Rendered Markdown Output:</h3>"))
    display(HTML('<div style="border: 2px solid #ccc; padding: 20px; border-radius: 5px; background-color: white;">'))
    display(Markdown(model_response))
    display(HTML('</div>'))
else:
    print("💡 Tip: Include 'markdown' in your prompt to see rendered output")

In [ ]:
# Display parsed structured data
if parsed_data:
    display(HTML("<h3>📊 Structured Data Extraction:</h3>"))
    
    # Count extracted fields
    extracted_fields = {k: v for k, v in parsed_data.items() if v and str(v).strip()}
    field_count = len(extracted_fields)
    
    print(f"📋 Successfully extracted {field_count} fields:")
    
    # Display in a formatted table
    html_table = """
    <table style="border-collapse: collapse; width: 100%; margin: 10px 0;">
        <tr style="background-color: #f0f0f0;">
            <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">Field</th>
            <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">Value</th>
        </tr>
    """
    
    for field, value in extracted_fields.items():
        html_table += f"""
        <tr>
            <td style="border: 1px solid #ddd; padding: 8px; font-weight: bold;">{field}</td>
            <td style="border: 1px solid #ddd; padding: 8px;">{value}</td>
        </tr>
        """
    
    html_table += "</table>"
    display(HTML(html_table))
    
    # Show extraction efficiency
    if ENABLE_DOCUMENT_AWARENESS and result:
        total_possible = result.get('fields_extracted', 25)
        extraction_rate = (field_count / total_possible) * 100 if total_possible > 0 else 0
        print(f"\n📈 Extraction Efficiency:")
        print(f"   Fields extracted: {field_count}/{total_possible}")
        print(f"   Success rate: {extraction_rate:.1f}%")
        print(f"   Document-specific schema: {result.get('detected_document_type', 'unknown')}")
        
else:
    print("❌ No structured data extracted")

## Comparison Results (if enabled)

In [ ]:
# Compare Phase 3 document-aware vs legacy unified processing
if result and ENABLE_DOCUMENT_AWARENESS:
    display(HTML("<h3>📊 Phase 3 vs Legacy Processing Comparison:</h3>"))
    
    # Extract Phase 3 metrics
    doc_type = result.get('detected_document_type', 'unknown')
    fields_used = result.get('fields_extracted', 25)
    efficiency_gain = result.get('efficiency_gain', '0%')
    field_reduction = result.get('field_reduction', 0)
    
    # Create comparison table
    html_comparison = f"""
    <table style="border-collapse: collapse; width: 100%; margin: 10px 0;">
        <tr style="background-color: #f0f0f0;">
            <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">Aspect</th>
            <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">Legacy (Phase 1)</th>
            <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">Document-Aware (Phase 3)</th>
            <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">Improvement</th>
        </tr>
        <tr>
            <td style="border: 1px solid #ddd; padding: 8px; font-weight: bold;">Document Detection</td>
            <td style="border: 1px solid #ddd; padding: 8px;">❌ None</td>
            <td style="border: 1px solid #ddd; padding: 8px;">✅ {doc_type}</td>
            <td style="border: 1px solid #ddd; padding: 8px;">Automatic classification</td>
        </tr>
        <tr>
            <td style="border: 1px solid #ddd; padding: 8px; font-weight: bold;">Schema Fields</td>
            <td style="border: 1px solid #ddd; padding: 8px;">25 (unified)</td>
            <td style="border: 1px solid #ddd; padding: 8px;">{fields_used} (targeted)</td>
            <td style="border: 1px solid #ddd; padding: 8px;">{field_reduction} fewer fields</td>
        </tr>
        <tr>
            <td style="border: 1px solid #ddd; padding: 8px; font-weight: bold;">Efficiency Gain</td>
            <td style="border: 1px solid #ddd; padding: 8px;">0% (baseline)</td>
            <td style="border: 1px solid #ddd; padding: 8px;">{efficiency_gain}</td>
            <td style="border: 1px solid #ddd; padding: 8px;">{'Significant' if efficiency_gain != '0%' else 'None'}</td>
        </tr>
        <tr>
            <td style="border: 1px solid #ddd; padding: 8px; font-weight: bold;">Prompt Optimization</td>
            <td style="border: 1px solid #ddd; padding: 8px;">❌ Generic</td>
            <td style="border: 1px solid #ddd; padding: 8px;">✅ Document-specific</td>
            <td style="border: 1px solid #ddd; padding: 8px;">Targeted extraction</td>
        </tr>
        <tr>
            <td style="border: 1px solid #ddd; padding: 8px; font-weight: bold;">Memory Usage</td>
            <td style="border: 1px solid #ddd; padding: 8px;">⚠️ Basic cleanup</td>
            <td style="border: 1px solid #ddd; padding: 8px;">✅ V100 optimized</td>
            <td style="border: 1px solid #ddd; padding: 8px;">Fragmentation prevention</td>
        </tr>
    </table>
    """
    
    display(HTML(html_comparison))
    
    # Summary
    print("🎯 Key Improvements:")
    if efficiency_gain != '0%':
        print(f"   • {efficiency_gain} efficiency improvement through field reduction")
    print(f"   • Automatic document type detection: {doc_type}")
    print(f"   • {field_reduction} fewer fields to process (25 → {fields_used})")
    print("   • V100-safe memory management with fragmentation prevention")
    print("   • Document-specific prompt optimization")
    
else:
    print("💡 Enable document awareness to see Phase 3 improvements")

## Phase 3 vs Legacy Comparison

This section shows the improvements from document-type-specific processing:

# Quick test with document-aware processing (edit and run)
QUICK_PROMPT = """
Extract all key business information from this document.
Return as structured JSON with proper field names.
Focus on the most relevant fields for this document type.
"""

print("🚀 Testing quick document-aware processing...")

try:
    # Create a simple custom extraction for quick testing
    if hasattr(processor, '_extract_with_custom_prompt'):
        # Use custom prompt extraction if available
        quick_response = processor._extract_with_custom_prompt(
            image_path, 
            QUICK_PROMPT,
            max_new_tokens=1024
        )
        quick_result = {
            'response': quick_response,
            'processing_time': 0  # Not tracked for quick test
        }
    else:
        # Fall back to full processing
        quick_result = processor.process_single_image(image_path)
        quick_result = {
            'response': quick_result.get('raw_response', ''),
            'processing_time': quick_result.get('processing_time', 0)
        }
    
    print(f"\n✅ Completed in {quick_result['processing_time']:.2f}s")
    print(f"📏 Response length: {len(quick_result['response'])} chars")
    print("\n📄 Output:")
    display(HTML(f'<div style="background-color: #f8f8f8; padding: 10px; border-radius: 5px; font-family: monospace; white-space: pre-wrap; max-height: 300px; overflow-y: auto;">{quick_result["response"]}</div>'))

except Exception as e:
    print(f"❌ Quick test failed: {e}")
    print("💡 This might be expected if custom prompt extraction is not available")

In [ ]:
# Test document-aware processing with different approaches
PROCESSING_VARIANTS = [
    ("Document-specific", "Auto-detect document type and extract relevant fields"),
    ("JSON focused", "Extract as clean JSON with proper field names and values"),
    ("Key-value pairs", "Extract as simple key: value pairs for this document type")
]

print("🔬 Testing document-aware processing variants...\n")

for i, (name, description) in enumerate(PROCESSING_VARIANTS, 1):
    print(f"Variant {i} ({name}): {description}")
    
    try:
        # Use full document-aware processing for each variant
        start_time = time.time()
        variant_result = processor.process_single_image(image_path)
        processing_time = time.time() - start_time
        
        response = variant_result.get('raw_response', '')
        doc_type = variant_result.get('detected_document_type', 'unknown')
        field_count = variant_result.get('fields_extracted', 0)
        
        print(f"   Time: {processing_time:.2f}s | Length: {len(response)} chars | Type: {doc_type} | Fields: {field_count}")
        
        # Show efficiency metrics if available
        efficiency = variant_result.get('efficiency_gain', '0%')
        if efficiency != '0%':
            print(f"   Efficiency gain: {efficiency}")
            
    except Exception as e:
        print(f"   ❌ Failed: {e}")
    
    print()

In [ ]:
# Test multiple prompts in sequence
PROMPT_VARIANTS = [
    "Convert to markdown.",
    "Extract as markdown with proper formatting.",
    "Create a markdown version of this document."
]

print("🔬 Testing prompt variants...\n")
for i, prompt in enumerate(PROMPT_VARIANTS, 1):
    print(f"Variant {i}: {prompt[:50]}...")
    result = tester.test_prompt(prompt, image_path)
    print(f"   Time: {result['processing_time']:.2f}s | Length: {len(result['response'])} chars\n")

In [ ]:
## Tips for Using This Document-Aware Notebook

### Phase 3 Document-Aware Features
This notebook now uses **Phase 3 document-type-specific processors** with advanced capabilities:

1. **Automatic Document Detection**: The system detects invoice, bank_statement, or receipt automatically
2. **Schema Optimization**: Uses 19 fields for invoices, 15 for statements/receipts (vs 25 unified)
3. **Targeted Extraction**: Prompts are optimized for each document type
4. **Efficiency Tracking**: Shows field reduction and performance gains

### Configuration Options
- **`ENABLE_DOCUMENT_AWARENESS`**: Toggle document-specific processing on/off
- **`SHOW_DOCUMENT_ANALYSIS`**: Display detection and efficiency metrics
- **`MODEL`**: Choose between "llama" and "internvl3" processors

### Usage Tips
1. **Edit the Configuration**: Modify `EXPERIMENTAL_PROMPT` and settings in the configuration cell
2. **Run All Cells**: Use Cell → Run All to execute the entire notebook
3. **Monitor Document Detection**: Check the analysis section to see detected document type
4. **Compare Efficiency**: Look for field reduction and efficiency gain percentages
5. **Structured Data**: Review the extracted fields table for quality assessment
6. **Quick Iteration**: Use the iteration cells for rapid testing

### Expected Performance
- **Document Detection**: 85%+ confidence for clear business documents
- **Field Reduction**: 24% (invoices), 40% (statements/receipts)
- **Processing Speed**: Faster due to fewer fields to extract
- **Memory Usage**: Optimized with V100-safe fragmentation handling

### Troubleshooting
- **Low Confidence Detection**: Document may be unclear or unsupported type (falls back to unified schema)
- **Memory Issues**: V100 optimizations are built-in, but reduce `MAX_TOKENS` if needed
- **Import Errors**: Ensure Phase 3 processors exist: `models/*_processor_v2.py`

### Keyboard Shortcuts
- `Shift + Enter`: Run cell and move to next
- `Ctrl + Enter`: Run cell and stay  
- `Alt + Enter`: Run cell and insert new cell below

## Tips for Using This Notebook

1. **Edit the Configuration**: Modify the `EXPERIMENTAL_PROMPT` in the configuration cell
2. **Run All Cells**: Use Cell → Run All to execute the entire notebook
3. **Quick Iteration**: Use the "Quick Iteration Zone" cells for rapid testing
4. **Compare Models**: Change `MODEL` between "llama" and "internvl3" to compare
5. **Adjust Tokens**: Increase `MAX_TOKENS` if output is being cut off
6. **Save Results**: The last cell exports your results to a markdown file

### Keyboard Shortcuts
- `Shift + Enter`: Run cell and move to next
- `Ctrl + Enter`: Run cell and stay
- `Alt + Enter`: Run cell and insert new cell below